# opentelemetry_exporter_http_transport

## Example 1: the built-in transport registry

In [ ]:
# Example 1: the two built-in names resolve to their classes without an entry-point scan; an unknown name is refused
from opentelemetry.exporter.http.transport import _load_http_transport_factory
from opentelemetry.exporter.http.transport._requests import RequestsHTTPTransport
from opentelemetry.exporter.http.transport._urllib3 import Urllib3HTTPTransport

assert _load_http_transport_factory("requests") is RequestsHTTPTransport
assert _load_http_transport_factory("urllib3") is Urllib3HTTPTransport
try:
    _load_http_transport_factory("carrier-pigeon")
except ValueError as error:
    assert "opentelemetry_http_transport" in str(error), error
    print("Example 1: unknown transport refused:", str(error)[:60], "...")
else:
    raise AssertionError("an unknown transport name was accepted")

## Example 2: the `requests` transport POSTs and reads the answer

In [ ]:
# Example 2: RequestsHTTPTransport POSTs bytes to a local echo server; status, JSON body and headers come back
import http.server, json, threading
from opentelemetry.exporter.http.transport._requests import RequestsHTTPTransport

class Echo(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        body = self.rfile.read(int(self.headers["Content-Length"]))
        out = json.dumps({"got": body.decode(), "type": self.headers["Content-Type"]}).encode()
        self.send_response(200); self.send_header("Content-Type", "application/json")
        self.send_header("X-Echo", "yes"); self.end_headers(); self.wfile.write(out)
    def log_message(self, *args): pass

server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), Echo)
threading.Thread(target=server.serve_forever, daemon=True).start()
transport = RequestsHTTPTransport(verify=True)
result = transport.request("POST", f"http://127.0.0.1:{server.server_address[1]}/v1/traces",
                           headers={"Content-Type": "application/x-protobuf"}, data=b"payload", timeout=5)
transport.close(); server.shutdown()
assert result.error is None and result.status_code == 200 and result.reason == "OK", result
assert result.json() == {"got": "payload", "type": "application/x-protobuf"} and result.headers()["x-echo"] == "yes"
print("Example 2:", result.status_code, result.json())

## Example 3: the `urllib3` transport does the same, with case-insensitive headers

In [ ]:
# Example 3: Urllib3HTTPTransport against the same kind of server; the header mapping ignores case
import http.server, json, threading
from opentelemetry.exporter.http.transport._urllib3 import Urllib3HTTPTransport

class Echo(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        body = self.rfile.read(int(self.headers["Content-Length"]))
        self.send_response(200); self.send_header("Content-Type", "application/json")
        self.send_header("X-Length", str(len(body))); self.end_headers()
        self.wfile.write(json.dumps({"got": body.hex()}).encode())
    def log_message(self, *args): pass

server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), Echo)
threading.Thread(target=server.serve_forever, daemon=True).start()
transport = Urllib3HTTPTransport(verify=True)
result = transport.request("POST", f"http://127.0.0.1:{server.server_address[1]}/v1/logs", data=b"\x00\x01\xff", timeout=5)
transport.close(); server.shutdown()
headers = result.headers()
assert result.status_code == 200 and result.json() == {"got": "0001ff"}, result
assert headers["x-length"] == headers["X-LENGTH"] == "3" and result.text() == '{"got": "0001ff"}'
print("Example 3:", result.status_code, result.text(), "| X-Length:", headers["x-length"])

## Example 4: a refused connection is a result with an error, never an exception

In [ ]:
# Example 4: nothing listens on the port, so both transports return a result whose error is a connection error
import socket
from opentelemetry.exporter.http.transport._requests import RequestsHTTPTransport
from opentelemetry.exporter.http.transport._urllib3 import Urllib3HTTPTransport

with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0)); port = probe.getsockname()[1]  # a port that is free once the socket closes
for transport in (RequestsHTTPTransport(), Urllib3HTTPTransport()):
    result = transport.request("POST", f"http://127.0.0.1:{port}/v1/traces", data=b"x", timeout=2)
    assert result.status_code is None and result.error is not None, result
    assert transport.is_connection_error(result.error), type(result.error)
    assert result.content() == b"" and not transport.is_connection_error(ValueError("other"))
    transport.close()
    print("Example 4:", type(transport).__name__, "->", type(result.error).__name__)

## Example 5: a 503 with Retry-After comes back as a status, not an error

In [ ]:
# Example 5: the server answers 503 + Retry-After; the result carries status and reason, and error stays None
import http.server, threading
from opentelemetry.exporter.http.transport._urllib3 import Urllib3HTTPTransport

class Busy(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        self.rfile.read(int(self.headers["Content-Length"]))
        self.send_response(503); self.send_header("Retry-After", "2"); self.send_header("Content-Length", "0")
        self.end_headers()
    def log_message(self, *args): pass

server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), Busy)
threading.Thread(target=server.serve_forever, daemon=True).start()
transport = Urllib3HTTPTransport()
result = transport.request("POST", f"http://127.0.0.1:{server.server_address[1]}/v1/metrics", data=b"m", timeout=5)
transport.close(); server.shutdown()
assert result.error is None and result.status_code == 503 and result.reason == "Service Unavailable", result
assert result.headers()["retry-after"] == "2" and result.content() == b""
print("Example 5:", result.status_code, result.reason, "| Retry-After:", result.headers()["retry-after"])